# Create Multiscale Zarr Pyramid for Web Visualization

This notebook builds a multiscale Zarr pyramid from the MODIS snow phenology Icechunk store so the dataset can be visualized as a slippy web map using [zarr-layer](https://github.com/carbonplan/zarr-layer) without creating a separate visualization copy or running a tile server.

**Stack**
- [topozarr](https://github.com/carbonplan/topozarr): generates coarsened multi-resolution levels following the [GeoZarr multiscales spec](https://github.com/zarr-developers/geozarr-spec)
- [zarr-layer](https://github.com/carbonplan/zarr-layer): TypeScript library that fetches and renders Zarr as a custom MapLibre/Mapbox layer, reprojecting on the GPU on the fly

**Why multiscales?**  
The full store is 86 400 × 43 200 pixels. Without overview levels, zarr-layer would need to fetch the entire array at every zoom level. Multiscales let the viewer fetch only the appropriate coarsened level for the current viewport — qualitatively matching the performance of a traditional tile server.

**Projection note**  
The store is in MODIS Sinusoidal, *not* Web Mercator. zarr-layer reprojects on the GPU client-side, so we can write the pyramid in native projection and preserve pixel fidelity.

---
**Dependencies**: `topozarr` and `xproj` are added to `pixi.toml` as PyPI dependencies. Run `pixi install` before launching this notebook.

## Imports

In [1]:
from pathlib import Path
import time
import sys
import pandas as pd
import xarray as xr
import numpy as np
import zarr
import icechunk
import rioxarray
import xproj
from topozarr import create_pyramid, ZarrLayerVarConfig
import dask
from dask.diagnostics import ProgressBar
import adlfs
from IPython.display import JSON
import json


sys.path.insert(0, str(Path('..').resolve()))
from modis_snow_phenology import Config

config = Config('config/config_with_secrets_v1.txt')

# note to self: READ https://zarr.readthedocs.io/en/latest/user-guide/performance/#concurrent-io-operations
# https://docs.dask.org/en/latest/scheduling.html
# from dask.distributed import Client
# client = Client() 
# https://github.com/carbonplan/topozarr/blob/main/scripts/build_demo_data.py
# from https://github.com/carbonplan/ocr/blob/main/ocr/pipeline/create_pyramid.py
# zarr.config.set({'async.concurrency': 128})
#dask.config.set(scheduler='threads')
dask.config.set(scheduler='threads',num_workers=32)
zarr.config.set({'async.concurrency': 128})

In [2]:
JSON(dask.config.config)

<IPython.core.display.JSON object>

/home/jovyan/repos/MODIS_snow_phenology/.pixi/envs/default/lib/python3.14/site-packages/jupyter_client/session.py:727: UserWarning: Message serialization failed with:
Out of range float values are not JSON compliant: inf
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant
  content = self.pack(content)


In [3]:
JSON(zarr.config.config)

<IPython.core.display.JSON object>

## 1. Open the Icechunk Store

In [4]:
storage = icechunk.azure_storage(
    account=config.AZURE_STORAGE_ACCOUNT,
    container=config.AZURE_CONTAINER,
    prefix=config.ICECHUNK_PREFIX,
    sas_token=config.AZURE_STORAGE_SAS_TOKEN,
)
repo = icechunk.Repository.open(storage)
session = repo.readonly_session('main')

ds = xr.open_zarr(session.store, zarr_format=3, consolidated=False, decode_coords='all',mask_and_scale=True)
ds

<xarray.Dataset> Size: 448GB
Dimensions:               (water_year: 10, y: 43200, x: 86400)
Coordinates:
  * water_year            (water_year) int64 80B 2015 2016 2017 ... 2023 2024
  * y                     (y) float64 346kB 1.001e+07 1.001e+07 ... -1.001e+07
  * x                     (x) float64 691kB -2.001e+07 -2.001e+07 ... 2.001e+07
    spatial_ref           int64 8B ...
Data variables:
    SAD_DOWY              (water_year, y, x) float32 149GB dask.array<chunksize=(1, 600, 600), meta=np.ndarray>
    SDD_DOWY              (water_year, y, x) float32 149GB dask.array<chunksize=(1, 600, 600), meta=np.ndarray>
    max_consec_snow_days  (water_year, y, x) float32 149GB dask.array<chunksize=(1, 600, 600), meta=np.ndarray>
Attributes:
    title:        Global MODIS Snow Phenology
    description:  Snow appearance date (SAD), snow disappearance date (SDD), ...
    source:       MODIS MOD10A2.061 via Microsoft Planetary Computer
    Conventions:  CF-1.8

## 2. Prepare Dataset

topozarr needs:
1. A CRS assigned via `xproj` (`.proj.assign_crs()`)
2. The `spatial_ref` scalar coordinate removed — topozarr manages CRS metadata internally and the scalar causes issues during coarsening

Data comes out of the store as `float32` (Zarr's `mask_and_scale` replaces `int16` fill values with `NaN`). `create_pyramid(method='mean')` propagates NaNs correctly, so no extra masking is needed.

In [5]:
# Extract CRS WKT from the rioxarray spatial_ref before we drop it
crs = ds.rio.crs
crs

CRS.from_wkt('PROJCS["unnamed",GEOGCS["Unknown datum based upon the custom spheroid",DATUM["Not specified (based on custom spheroid)",SPHEROID["Custom spheroid",6371007.181,0]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Sinusoidal"],PARAMETER["longitude_of_center",0],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["Meter",1],AXIS["Easting",EAST],AXIS["Northing",NORTH]]')

In [6]:
# Drop the CF-convention spatial_ref scalar — xproj will carry the CRS instead
ds_clean = ds.drop_vars('spatial_ref')

# Assign CRS via xproj so topozarr can find it
ds_crs = ds_clean.proj.assign_crs(spatial_ref_crs={'wkt': crs.to_wkt()})
ds_crs

<xarray.Dataset> Size: 448GB
Dimensions:               (water_year: 10, y: 43200, x: 86400)
Coordinates:
  * water_year            (water_year) int64 80B 2015 2016 2017 ... 2023 2024
  * y                     (y) float64 346kB 1.001e+07 1.001e+07 ... -1.001e+07
  * x                     (x) float64 691kB -2.001e+07 -2.001e+07 ... 2.001e+07
  * wkt                   int64 8B 0
Data variables:
    SAD_DOWY              (water_year, y, x) float32 149GB dask.array<chunksize=(1, 600, 600), meta=np.ndarray>
    SDD_DOWY              (water_year, y, x) float32 149GB dask.array<chunksize=(1, 600, 600), meta=np.ndarray>
    max_consec_snow_days  (water_year, y, x) float32 149GB dask.array<chunksize=(1, 600, 600), meta=np.ndarray>
Indexes:
    wkt      CRSIndex (crs=PROJCS["unnamed",GEOGCS["Unknown datum based upon the custom sphero ...)
Attributes:
    title:        Global MODIS Snow Phenology
    description:  Snow appearance date (SAD), snow disappearance date (SDD), ...
    source:       MODIS MOD10A2.061 via Microsoft Planetary Computer
    Conventions:  CF-1.8

## 3. Create Multiscale Pyramid

### Choosing the number of levels

Each level coarsens by 2× in both spatial dimensions. In topozarr's convention **level 0 is the finest (native resolution)** and level N_LEVELS−1 is the coarsest. The coarsest level should be small enough to fetch in a single request at the lowest zoom.

| N_LEVELS | Coarsest level | Coarsest shape (y × x) | Coarsest pixel size |
|----------|---------------|------------------------|---------------------|
| 7        | level 6       | 337 × 675              | ~59 km              |
| 8        | level 7       | 169 × 337              | ~118 km             |
| 9        | level 8       | 84 × 169               | ~236 km             |

8 levels gives a manageable coarsest tile (~169 × 337 pixels) while still preserving meaningful spatial detail at the finest level.

### Coarsening method
`method='mean'` is used for all three variables. For day-of-year metrics (SAD, SDD) this is a spatial average, which is visually reasonable. Override with `'min'` or `'max'` if you want conservative estimates.

### Non-spatial dimension
The `water_year` dimension is preserved at every pyramid level — zarr-layer handles it as a non-spatial dimension that the user can slice interactively.

In [7]:
# SAD/SDD valid range is 1–366 (day of water year); max_consec is 0–366
layer_hints = {
    'SAD_DOWY': ZarrLayerVarConfig(clim=[1, 366],  colormap='purples'),
    'SDD_DOWY': ZarrLayerVarConfig(clim=[1, 366],  colormap='reds'),
    'max_consec_snow_days': ZarrLayerVarConfig(clim=[0, 366], colormap='blues'), 
}

In [8]:
N_LEVELS = 8

pyramid = create_pyramid(
    ds_crs,
    levels=N_LEVELS,
    x_dim='x',
    y_dim='y',
    method='mean',
    target_chunk_bytes=int(0.5 * 1024 * 1024),  # web friendly 500 KB chunks
    chunks_per_shard=4,
    layer_hints=layer_hints,
)

pyramid

Pyramid(datatree=<xarray.DataTree 'root'>
Group: /
│   Attributes:
│       zarr_conventions:    [{'schema_url': 'https://raw.githubusercontent.com/z...
│       multiscales:         {'layout': [{'asset': '0', 'transform': {'scale': [1...
│       proj:code:           PROJCS["unnamed",GEOGCS["Unknown datum based upon th...
│       spatial:dimensions:  ['y', 'x']
│       spatial:transform:   [463.3127165287733, 0.0, -20015109.354005992, 0.0, -...
│       spatial:bbox:        [-20015109.354005992, -10007554.677040007, 20015109....
│       spatial:shape:       [43200, 86400]
│       zarr-layer:          {'SAD_DOWY': {'clim': [1, 366], 'colormap': 'purples...
├── Group: /0
│       Dimensions:               (water_year: 10, y: 43200, x: 86400)
│       Coordinates:
│         * water_year            (water_year) int64 80B 2015 2016 2017 ... 2023 2024
│         * y                     (y) float64 346kB 1.001e+07 1.001e+07 ... -1.001e+07
│         * x                     (x) float64 691kB -2.001e+

In [9]:
# maybe here we can update the encoding?
int16_fill_value = np.iinfo(np.int16).min  # -32768
INT16_ENCODING = {
    'dtype': 'int16',
    '_FillValue': int16_fill_value,
    'write_empty_chunks': False,
}

# Merge with topozarr's chunk/shard encoding (which only has 'chunks'/'shards')
for level_path, level_enc in pyramid.encoding.items():
    for var_name in level_enc:
        level_enc[var_name].update(INT16_ENCODING)

In [10]:
pyramid.encoding

{'/0': {'SAD_DOWY': {'chunks': (1, 360, 362),
   'shards': (1, 1440, 1448),
   'dtype': 'int16',
   '_FillValue': -32768,
   'write_empty_chunks': False},
  'SDD_DOWY': {'chunks': (1, 360, 362),
   'shards': (1, 1440, 1448),
   'dtype': 'int16',
   '_FillValue': -32768,
   'write_empty_chunks': False},
  'max_consec_snow_days': {'chunks': (1, 360, 362),
   'shards': (1, 1440, 1448),
   'dtype': 'int16',
   '_FillValue': -32768,
   'write_empty_chunks': False}},
 '/1': {'SAD_DOWY': {'chunks': (1, 360, 360),
   'shards': (1, 1440, 1440),
   'dtype': 'int16',
   '_FillValue': -32768,
   'write_empty_chunks': False},
  'SDD_DOWY': {'chunks': (1, 360, 360),
   'shards': (1, 1440, 1440),
   'dtype': 'int16',
   '_FillValue': -32768,
   'write_empty_chunks': False},
  'max_consec_snow_days': {'chunks': (1, 360, 360),
   'shards': (1, 1440, 1440),
   'dtype': 'int16',
   '_FillValue': -32768,
   'write_empty_chunks': False}},
 '/2': {'SAD_DOWY': {'chunks': (1, 360, 360),
   'shards': (1, 1440,

## 4. Write Pyramid to Zarr

- **Azure**: write to the `uwcryo` storage account under a new prefix so zarr-layer can reach it from the browser

The DataTree hierarchy maps directly to Zarr group hierarchy. In Zarr v3 user attributes live in `zarr.json` (not `.zattrs`):
```
modis_snow_phenology_multiscale.zarr/
├── zarr.json        ← multiscales + layer-hints metadata (Zarr v3)
├── 0/               ← finest level (86 400 × 43 200, native resolution)
├── 1/
├── ...
└── 7/               ← coarsest level (~675 × 337)
```

### Write to Azure as Plain Zarr v3

Write via `adlfs.AzureBlobFileSystem` + `fs.get_mapper()` — no Icechunk. The pyramid is a derived, read-only product, so versioning adds no value. Using a plain store also eliminates the Icechunk manifest lookup on every chunk fetch, halving HTTP round-trips for the web map.

For zarr-layer to reach the store from a browser the container must allow **anonymous blob reads** or use a SAS URL. Set the container access level to *Blob* in the Azure portal (or via `az storage container set-permission`).

In [11]:
MULTISCALE_PREFIX = 'modis_snow_phenology/modis_snow_phenology_multiscale_v1'
MULTISCALE_ROOT_PATH = f"{config.AZURE_CONTAINER}/{MULTISCALE_PREFIX}"

fs = adlfs.AzureBlobFileSystem(
    account_name=config.AZURE_STORAGE_ACCOUNT,
    sas_token=config.AZURE_STORAGE_SAS_TOKEN,
    #asynchronous=True,
    skip_instance_cache=True,
)

In [12]:
# clean up any existing store at the target location before creating new one. have to use await because of asychronous=True
remove_existing_store = True  # set to True to delete existing store and start fresh (warning: this will delete all existing data in the store!)
if remove_existing_store == True:
    if fs.exists(MULTISCALE_ROOT_PATH):
        fs.rm(MULTISCALE_ROOT_PATH, recursive=True)
        print(f"Deleted existing store at {MULTISCALE_ROOT_PATH}")
    else:
        print("No existing store found, nothing to delete")

Deleted existing store at snowmelt/modis_snow_phenology/modis_snow_phenology_multiscale_v1


In [ ]:
# def write_pyramid(pyramid, store, x_dim='x', y_dim='y', method='mean', zarr_format=3):

#     def _level_store(level):
#         try:
#             from fsspec.mapping import FSMap
#             if isinstance(store, FSMap):
#                 return store.fs.get_mapper(store.root.rstrip('/') + '/' + str(level)), None
#         except ImportError:
#             pass
#         return store, str(level)

#     def _write(ds, level, enc):
#         s, g = _level_store(level)
#         kw = {} if g is None else {'group': g}
#         ds.to_zarr(s, mode='a', encoding=enc, zarr_format=zarr_format,
#                    consolidated=False, **kw)

#     def _open(level):
#         s, g = _level_store(level)
#         kw = {} if g is None else {'group': g}
#         return xr.open_zarr(s, consolidated=False, **kw)

#     n_levels = len(pyramid.encoding)

#     root = zarr.open_group(store, mode='a', zarr_format=zarr_format)
#     root.attrs.update(pyramid.dt.attrs)

#     ds0 = pyramid.dt['/0'].ds
#     ny, nx = ds0.sizes.get(y_dim, '?'), ds0.sizes.get(x_dim, '?')
#     print(f'[1/{n_levels}] Writing level 0  ({ny} x {nx})', flush=True)
#     t0 = time.perf_counter()
#     _write(ds0, 0, pyramid.encoding['/0'])
#     print(f'[1/{n_levels}] Level 0 done  ({time.perf_counter() - t0:.1f}s)', flush=True)

#     for i in range(1, n_levels):
#         template_ds = pyramid.dt[f'/{i}'].ds
#         prev_ds = _open(i - 1)
#         curr_ds = getattr(prev_ds.coarsen({x_dim: 2, y_dim: 2}, boundary='trim'), method)()
#         curr_ds.attrs = template_ds.attrs
#         for var in curr_ds.data_vars:
#             if var in template_ds:
#                 curr_ds[var].attrs = template_ds[var].attrs
#         ny, nx = template_ds.sizes.get(y_dim, '?'), template_ds.sizes.get(x_dim, '?')
#         print(f'[{i + 1}/{n_levels}] Writing level {i}  ({ny} x {nx})', flush=True)
#         t0 = time.perf_counter()
#         _write(curr_ds, i, pyramid.encoding[f'/{i}'])
#         print(f'[{i + 1}/{n_levels}] Level {i} done  ({time.perf_counter() - t0:.1f}s)', flush=True)

def write_pyramid(pyramid, fs, path, x_dim="x", y_dim="y", method="mean", zarr_format=3):
    root_path = path.rstrip("/")
    n_levels = len(pyramid.encoding)

    root_store = fs.get_mapper(root_path)
    root = zarr.open_group(root_store, mode="w", zarr_format=zarr_format)
    root.attrs.update(pyramid.dt.attrs)

    ds0 = pyramid.dt["/0"].ds
    ny, nx = ds0.sizes.get(y_dim, "?"), ds0.sizes.get(x_dim, "?")
    print(f"[1/{n_levels}] Writing level 0  ({ny} x {nx})", flush=True)
    t0 = time.perf_counter()
    store0 = fs.get_mapper(f"{root_path}/0")
    ds0.to_zarr(store=store0, mode="a", encoding=pyramid.encoding["/0"],
                zarr_format=zarr_format, consolidated=False)
    #zarr.consolidate_metadata(store0)
    print(f"[1/{n_levels}] Level 0 done  ({time.perf_counter() - t0:.1f}s)", flush=True)

    for i in range(1, n_levels):
        template_ds = pyramid.dt[f"/{i}"].ds
        prev_store = fs.get_mapper(f"{root_path}/{i - 1}")
        current_store = fs.get_mapper(f"{root_path}/{i}")

        prev_ds = xr.open_zarr(prev_store, consolidated=False, mask_and_scale=True)
        curr_ds = prev_ds.coarsen(dim={x_dim: 2, y_dim: 2}, boundary="trim", coord_func="mean")
        curr_ds.attrs = template_ds.attrs
        for var in curr_ds.data_vars:
            if var in template_ds:
                curr_ds[var].attrs = template_ds[var].attrs

        ny, nx = template_ds.sizes.get(y_dim, "?"), template_ds.sizes.get(x_dim, "?")
        print(f"[{i + 1}/{n_levels}] Writing level {i}  ({ny} x {nx})", flush=True)
        t0 = time.perf_counter()
        curr_ds.to_zarr(store=current_store, mode="a", encoding=pyramid.encoding[f"/{i}"],
                        zarr_format=zarr_format, consolidated=False)
        #zarr.consolidate_metadata(current_store)
        print(f"[{i + 1}/{n_levels}] Level {i} done  ({time.perf_counter() - t0:.1f}s)", flush=True)

In [15]:
%%time
#root_store = fs.get_mapper(MULTISCALE_ROOT_PATH)
with ProgressBar(dt=30):
    write_pyramid(
        pyramid=pyramid,
        fs=fs,
        path = MULTISCALE_ROOT_PATH,
        x_dim='x',
        y_dim='y',
        method='mean',
        zarr_format=3,
    )

[1/8] Writing level 0  (43200 x 86400)
[########################################] | 100% Completed | 45m 9ss
[1/8] Level 0 done  (2742.8s)
CPU times: user 53min 3s, sys: 5min 32s, total: 58min 36s
Wall time: 45min 48s


ValueError: Window dimensions ('x', 'y') not found in Dataset dimensions ()

In [ ]:
n_levels = len(pyramid.encoding)

In [ ]:
zarr_format=3
root_store = fs.get_mapper(MULTISCALE_ROOT_PATH)


In [ ]:
root = zarr.open_group(store, mode='a', zarr_format=zarr_format)
root

In [ ]:
root.attrs.update(pyramid.dt.attrs)

In [ ]:
ds0 = pyramid.dt['/0'].ds
ds0    

In [ ]:
ny, nx = ds0.sizes.get(y_dim, '?'), ds0.sizes.get(x_dim, '?')

In [ ]:
store0 = fs.get_mapper(MULTISCALE_ROOT_PATH+f'/{level}')
ds0.to_zarr(store=store0, encoding=pyramid.encoding['/0'])

In [ ]:
prev_store = fs.get_mapper(MULTISCALE_ROOT_PATH+f'/{level-1}')
current_store = fs.get_mapper(MULTISCALE_ROOT_PATH+f'/{level}')

prev_ds = xr.open_zarr(store=prev_store, consolidated=False, mask_and_scale=True, zarr_format=3) # decode_coords='all'>
current_ds = prev_ds.coarsen({x_dim: 2, y_dim: 2}, boundary='trim', method='mean')
current_ds.to_zarr(store=current_store, consolidated=False, encoding=pyramid.encoding[f'/{level}'], zarr_format=3)

In [ ]:
%%time
root_store = fs.get_mapper(MULTISCALE_ROOT_PATH)
with ProgressBar(dt=30):
    write_pyramid(
        pyramid=pyramid,
        store=root_store,
        x_dim='x',
        y_dim='y',
        method='mean',
        zarr_format=3,
    )

### Set Cache-Control headers on all blobs

Set `Cache-Control: public, max-age=31536000` on every blob so browsers and CDNs cache chunks aggressively.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from azure.storage.blob import BlobServiceClient, ContentSettings

blob_service = BlobServiceClient(
    account_url=f'https://{config.AZURE_STORAGE_ACCOUNT}.blob.core.windows.net',
    credential=config.AZURE_STORAGE_SAS_TOKEN,
)
container_client = blob_service.get_container_client(config.AZURE_CONTAINER)

def set_cache_control(blob):
    container_client.get_blob_client(blob.name).set_http_headers(
        ContentSettings(cache_control='public, max-age=31536000')
    )

prefix = MULTISCALE_PREFIX + '/'
blobs = list(container_client.list_blobs(name_starts_with=prefix))

with ThreadPoolExecutor(max_workers=32) as executor:
    futures = {executor.submit(set_cache_control, b): b for b in blobs}
    for i, f in enumerate(as_completed(futures), 1):
        f.result()  # raise if any failed
        print(f'{i}/{len(blobs)} done', end='\r')
print(f'\nSet Cache-Control on {len(blobs)} blobs')

## Code graveyard

In [ ]:
# # diagnose issues
# store = fs.get_mapper(MULTISCALE_PATH)
# # pyramid.dt.to_zarr(store, mode='w', encoding=pyramid.encoding, zarr_format=3, consolidated=False)

# test_mapper = store.fs.get_mapper(store.root.rstrip('/') + '/test')
# import xarray as xr, numpy as np
# xr.Dataset({'a': xr.DataArray(np.ones((4,4)), dims=['y','x'])}).to_zarr(
#     test_mapper, mode='a', zarr_format=3, consolidated=False
# )
# print("files on azure:", store.fs.ls(test_mapper.root))

# import zarr
# grp = zarr.open_group(test_mapper, mode='r')
# print("zarr keys:", list(grp.keys()))
# print("xarray:", xr.open_zarr(test_mapper, consolidated=False))

# # Does the array metadata exist?
# print(store.fs.ls(test_mapper.root + '/a'))

# # Try mode='w' for the level write (not the root — just the level sub-mapper)
# xr.Dataset({'a': xr.DataArray(np.ones((4,4)), dims=['y','x'])}).to_zarr(
#     test_mapper, mode='w', zarr_format=3, consolidated=False
# )
# grp = zarr.open_group(test_mapper, mode='r')
# print("keys:", list(grp.keys()))


# store.fs.invalidate_cache()  # no argument = clear entire dircache

# grp = zarr.open_group(test_mapper, mode='r')
# print("keys:", list(grp.keys()))

# grp = zarr.open_group(test_mapper, mode='r')
# print("keys:", list(grp.keys()))
# try:
#     print("grp['a']:", grp['a'])
# except Exception as e:
#     print("direct access error:", e)

# import adlfs
# fs_fresh = adlfs.AzureBlobFileSystem(
#     account_name=config.AZURE_STORAGE_ACCOUNT,
#     sas_token=config.AZURE_STORAGE_SAS_TOKEN,
# )
# mapper_fresh = fs_fresh.get_mapper(test_mapper.root)
# grp_fresh = zarr.open_group(mapper_fresh, mode='r')
# print("fresh fs keys:", list(grp_fresh.keys()))

# result = store.fs.ls(test_mapper.root, detail=True)
# for item in result:
#     print(item)

# store.fs.ls(test_mapper.root)  # warm cache
# grp = zarr.open_group(test_mapper, mode='r')
# print("keys:", list(grp.keys()))





# # write something fresh
# test_mapper = store.fs.get_mapper(store.root.rstrip('/') + '/test')
# import xarray as xr, numpy as np
# xr.Dataset({'a': xr.DataArray(np.ones((4,4)), dims=['y','x'])}).to_zarr(test_mapper, mode='a', zarr_format=3, consolidated=False)

# # read without cache invalidation - should be empty
# print(xr.open_zarr(test_mapper, consolidated=False))

# # invalidate and retry - should work
# test_mapper.fs.invalidate_cache(test_mapper.root)
# print(xr.open_zarr(test_mapper, consolidated=False))

# test_mapper = store.fs.get_mapper(store.root.rstrip('/') + '/test')
# import xarray as xr, numpy as np
# xr.Dataset({'a': xr.DataArray(np.ones((4,4)), dims=['y','x'])}).to_zarr(
#     test_mapper, mode='w', zarr_format=3, consolidated=False
# )
# print("files on azure:", store.fs.ls(test_mapper.root))

# import zarr
# grp = zarr.open_group(test_mapper, mode='r')
# print("zarr keys:", list(grp.keys()))
# print("xarray:", xr.open_zarr(test_mapper, consolidated=False))

# print(store.fs.ls(test_mapper.root + '/a'))


# # Try mode='w' for the level write (not the root — just the level sub-mapper)
# xr.Dataset({'a': xr.DataArray(np.ones((4,4)), dims=['y','x'])}).to_zarr(
#     test_mapper, mode='w', zarr_format=3, consolidated=False
# )
# grp = zarr.open_group(test_mapper, mode='r')
# print("keys:", list(grp.keys()))

In [ ]:
# rows = []
# variables = ["SAD_DOWY", "SDD_DOWY", "max_consec_snow_days"]

# for level in pyramid.dt:
#     sizes = dict(pyramid.dt[level].ds.sizes)
#     for var in variables:
#         row = {
#             "Level": level,
#             "Dataset Sizes": str(sizes),
#             "Variable": var,
#         }

#         encoding = pyramid.dt[level].ds[var].encoding
#         clean_encoding = {k: str(v) for k, v in encoding.items()}

#         row.update(clean_encoding)
#         rows.append(row)

# encoding_df = pd.DataFrame(rows)
# encoding_df.set_index(["Level", "Dataset Sizes", "Variable"], inplace=True)
# with pd.option_context("display.max_colwidth", None):
#     display(encoding_df)

# %%time
# store = fs.get_mapper(MULTISCALE_PATH)

# from dask.diagnostics import ProgressBar
# with ProgressBar(dt=30):
#     pyramid.dt.to_zarr(store, 
#                        mode='w', 
#                        encoding=pyramid.encoding, 
#                        zarr_format=3, 
#                        consolidated=False,
#                        # align_chunks=False, could try align_chunks=False from https://github.com/carbonplan/ocr/blob/main/ocr/pipeline/create_pyramid.py
#                       )

# AZURE_URL = (
#     f'https://{config.AZURE_STORAGE_ACCOUNT}.blob.core.windows.net'
#     f'/{MULTISCALE_PATH}'
# )
# print(f'Written to: {AZURE_URL}')

# %%time
# store = fs.get_mapper(MULTISCALE_PATH)

# with ProgressBar(dt=30):
#     write_pyramid(
#         pyramid=pyramid,
#         store=store,
#         x_dim='x',
#         y_dim='y',
#         method='mean',
#         mode='w',
#         zarr_format=3,
#     )

# def level_store(store, level):
#     try:
#         from fsspec.mapping import FSMap
#         if isinstance(store, FSMap):
#             root = store.root.rstrip('/')
#             return store.fs.get_mapper(f'{root}/{level}'), None
#     except ImportError:
#         pass
#     return store, str(level)
        
# def write_pyramid(pyramid, store, x_dim='x', y_dim='y', method='mean',
#                   mode='w', zarr_format=3):
    
#     n_levels = len(pyramid.encoding)

#     root = zarr.open_group(store, mode=mode, zarr_format=zarr_format)
#     root.attrs.update(pyramid.dt.attrs)

#     level_store, group = level_store(store, 0)
#     pyramid.dt['/0'].ds.to_zarr(
#         level_store, group=group, mode='a',
#         encoding=pyramid.encoding['/0'],
#         zarr_format=zarr_format, consolidated=False,
#     )

#     for i in range(1, n_levels):
#         template_ds = pyramid.dt[f'/{i}'].ds
#         prev_store, prev_group = level_store(store, i - 1)
#         prev_ds = xr.open_zarr(prev_store, group=prev_group, consolidated=False)

#         curr_ds = getattr(
#             prev_ds.coarsen({x_dim: 2, y_dim: 2}, boundary='trim'), method
#         )()
#         curr_ds.attrs = template_ds.attrs
#         for var in curr_ds.data_vars:
#             if var in template_ds:
#                 curr_ds[var].attrs = template_ds[var].attrs

#         curr_store, curr_group = level_store(store, i)
#         curr_ds.to_zarr(
#             curr_store, group=curr_group, mode='a',
#             encoding=pyramid.encoding[f'/{i}'],
#             zarr_format=zarr_format, consolidated=False,
#         )
# store = fs.get_mapper(MULTISCALE_PATH)
# level_store, group = level_store(store, 0)
# prev_store, prev_group = level_store, group
# prev_ds = xr.open_zarr(prev_store, group=prev_group, consolidated=False)
# prev_ds

# def write_pyramid(pyramid, store, x_dim='x', y_dim='y', method='mean',
#                   mode='w', zarr_format=3):

#     def _is_fsmap(store):
#         try:
#             from fsspec.mapping import FSMap
#             return isinstance(store, FSMap)
#         except ImportError:
#             return False

#     def _level_store(level):
#         # zarr v3 rejects group= with FSMap; build a FsspecStore for the subpath instead.
#         if _is_fsmap(store):
#             return zarr.storage.FsspecStore(
#                 store.fs, path=store.root.rstrip('/') + '/' + str(level)
#             )
#         return None  # use group= parameter below

#     def _write_level(ds, level, enc):
#         sub = _level_store(level)
#         if sub is not None:
#             ds.to_zarr(sub, mode='a', encoding=enc,
#                        zarr_format=zarr_format, consolidated=False)
#         else:
#             ds.to_zarr(store, group=str(level), mode='a', encoding=enc,
#                        zarr_format=zarr_format, consolidated=False)

#     def _open_level(level):
#         sub = _level_store(level)
#         if sub is not None:
#             return xr.open_zarr(sub, consolidated=False)
#         return xr.open_zarr(store, group=str(level), consolidated=False)

#     n_levels = len(pyramid.encoding)
#     root = zarr.open_group(store, mode=mode, zarr_format=zarr_format)
#     root.attrs.update(pyramid.dt.attrs)

#     ds0 = pyramid.dt['/0'].ds
#     ny, nx = ds0.sizes.get(y_dim, '?'), ds0.sizes.get(x_dim, '?')
#     print(f'[1/{n_levels}] Writing level 0  ({ny} x {nx})', flush=True)
#     t0 = time.perf_counter()
#     _write_level(ds0, 0, pyramid.encoding['/0'])
#     print(f'[1/{n_levels}] Level 0 done  ({time.perf_counter()-t0:.1f}s)', flush=True)

#     for i in range(1, n_levels):
#         template_ds = pyramid.dt[f'/{i}'].ds
#         prev_ds = _open_level(i - 1)
#         curr_ds = getattr(
#             prev_ds.coarsen({x_dim: 2, y_dim: 2}, boundary='trim'), method
#         )()
#         curr_ds.attrs = template_ds.attrs
#         for var in curr_ds.data_vars:
#             if var in template_ds:
#                 curr_ds[var].attrs = template_ds[var].attrs
#         ny, nx = template_ds.sizes.get(y_dim, '?'), template_ds.sizes.get(x_dim, '?')
#         print(f'[{i+1}/{n_levels}] Writing level {i}  ({ny} x {nx})', flush=True)
#         t0 = time.perf_counter()
#         _write_level(curr_ds, i, pyramid.encoding[f'/{i}'])
#         print(f'[{i+1}/{n_levels}] Level {i} done  ({time.perf_counter()-t0:.1f}s)', flush=True)


# def write_pyramid(pyramid, fs, path, x_dim='x', y_dim='y', method='mean',
#                   mode='w', zarr_format=3, consolidated=False):

#     root_path = path.rstrip('/')

#     def _level_mapper(level):
#         return fs.get_mapper(f'{root_path}/{level}')

#     def _write_level(ds, level, enc):
#         ds.to_zarr(_level_mapper(level), mode='a', encoding=enc,
#                    zarr_format=zarr_format, consolidated=False)

#     def _open_level(level):
#         return xr.open_zarr(_level_mapper(level), consolidated=False)

#     n_levels = len(pyramid.encoding)

#     root = zarr.open_group(fs.get_mapper(root_path), mode=mode, zarr_format=zarr_format)
#     root.attrs.update(pyramid.dt.attrs)

#     ds0 = pyramid.dt['/0'].ds
#     ny, nx = ds0.sizes.get(y_dim, '?'), ds0.sizes.get(x_dim, '?')
#     print(f'[1/{n_levels}] Writing level 0  ({ny} x {nx})', flush=True)
#     t0 = time.perf_counter()
#     _write_level(ds0, 0, pyramid.encoding['/0'])
#     print(f'[1/{n_levels}] Level 0 done  ({time.perf_counter() - t0:.1f}s)', flush=True)

#     for i in range(1, n_levels):
#         template_ds = pyramid.dt[f'/{i}'].ds
#         prev_ds = _open_level(i - 1)
#         curr_ds = getattr(prev_ds.coarsen({x_dim: 2, y_dim: 2}, boundary='trim'), method)()
#         curr_ds.attrs = template_ds.attrs
#         for var in curr_ds.data_vars:
#             if var in template_ds:
#                 curr_ds[var].attrs = template_ds[var].attrs
#         ny, nx = template_ds.sizes.get(y_dim, '?'), template_ds.sizes.get(x_dim, '?')
#         print(f'[{i + 1}/{n_levels}] Writing level {i}  ({ny} x {nx})', flush=True)
#         t0 = time.perf_counter()
#         _write_level(curr_ds, i, pyramid.encoding[f'/{i}'])
#         print(f'[{i + 1}/{n_levels}] Level {i} done  ({time.perf_counter() - t0:.1f}s)', flush=True)